### Getting Started 

#### Install Google Gen AI SDK for Python

In [1]:
%pip install --upgrade --quiet google-genai

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-aiplatform 1.130.0 requires google-genai<2.0.0,>=1.37.0, but you have google-genai 2.12.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


#### Restart runtime
To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which restarts the current kernel.

In [2]:
# restart the kernel after libraries are loaded
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

### Set Google Cloud project information and create client

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).



In [1]:
# Define project information
PROJECT_ID = "qwiklabs-gcp-02-73e0cf438d4c"  # @param {type:"string"}
LOCATION = "global"

# Create the API client
from google import genai
client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)

#### Import libraries


In [2]:
from google.genai.types import (
    FunctionDeclaration,
    GenerateContentConfig,
    Tool,
    Part
)

### Task 3. Create a function call using Gemini

In [3]:
# Task 3.1
# use the following documentation to assist you complete this cell
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling
# Load Gemini 3.5 Flash Model
model_id = "gemini-3.5-flash"

In [6]:
# Task 3.2
# use the following documentation to assist you complete this cell
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling
get_current_weather_func = FunctionDeclaration(
    name="get_current_weather",
    description="Get the current weather in a given location",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "Location"
            }
        }
    },
)

In [7]:
# Task 3.3
# use the following documentation to assist you complete this cell
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling
weather_tool = Tool(
    function_declarations=[get_current_weather_func],
)

In [11]:
# Task 3.4
# use the following documentation to assist you complete this cell
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling
prompt = "What is the weather like in Boston?"
response = client.models.generate_content(
    model=model_id,
    contents=prompt,
    config=GenerateContentConfig(
        tools=[weather_tool],
        temperature=0,
    ),
)
response.model_dump_json(indent=2, exclude_none=True)

'{\n  "sdk_http_response": {\n    "headers": {\n      "content-type": "application/json; charset=UTF-8",\n      "vary": "Origin, X-Origin, Referer",\n      "content-encoding": "gzip",\n      "date": "Fri, 17 Jul 2026 09:23:38 GMT",\n      "server": "scaffolding on HTTPServer2",\n      "x-xss-protection": "0",\n      "x-frame-options": "SAMEORIGIN",\n      "x-content-type-options": "nosniff",\n      "transfer-encoding": "chunked"\n    }\n  },\n  "candidates": [\n    {\n      "content": {\n        "parts": [\n          {\n            "function_call": {\n              "args": {\n                "location": "Boston"\n              },\n              "name": "get_current_weather"\n            },\n            "thought_signature": "AY89a19qrkI8OQUpVhC_wFr1eSqB0LJmPlUjgz2lvdvUkHLqB_BSjWB9cpVz6gx5MTw="\n          }\n        ],\n        "role": "model"\n      },\n      "finish_reason": "STOP"\n    }\n  ],\n  "create_time": "2026-07-17T09:23:37.581405Z",\n  "model_version": "gemini-3.5-flash",\n  

### Task 4. Describe video contents using Gemini

In [12]:
# Run the following cell to import required libraries 
from google.genai.types import (
    GenerationConfig,
    Image,
    Part,
)

In [13]:
# Task 4.1
# Load the correct Gemini model use the following documentation to assist:
# https://cloud.google.com/vertex-ai/docs/generative-ai/multimodal/overview#supported-use-cases
# Load Gemini 3.5 Flash Model
multimodal_model = "gemini-3.5-flash"

In [14]:
import http.client
import typing
import urllib.request

import IPython.display
from PIL import Image as PIL_Image
from PIL import ImageOps as PIL_ImageOps


def display_images(
    images: typing.Iterable[Image],
    max_width: int = 600,
    max_height: int = 350,
) -> None:
    for image in images:
        pil_image = typing.cast(PIL_Image.Image, image._pil_image)
        if pil_image.mode != "RGB":
            # RGB is supported by all Jupyter environments (e.g. RGBA is not yet)
            pil_image = pil_image.convert("RGB")
        image_width, image_height = pil_image.size
        if max_width < image_width or max_height < image_height:
            # Resize to display a smaller notebook image
            pil_image = PIL_ImageOps.contain(pil_image, (max_width, max_height))
        IPython.display.display(pil_image)


def get_image_bytes_from_url(image_url: str) -> bytes:
    with urllib.request.urlopen(image_url) as response:
        response = typing.cast(http.client.HTTPResponse, response)
        image_bytes = response.read()
    return image_bytes


def load_image_from_url(image_url: str) -> Image:
    image_bytes = get_image_bytes_from_url(image_url)
    return Image.from_bytes(image_bytes)


def display_content_as_image(content: str | Image | Part) -> bool:
    if not isinstance(content, Image):
        return False
    display_images([content])
    return True


def display_content_as_video(content: str | Image | Part) -> bool:
    if not isinstance(content, Part):
        return False
    part = typing.cast(Part, content)
    file_path = part.file_data.file_uri.removeprefix("gs://")
    video_url = f"https://storage.googleapis.com/{file_path}"
    IPython.display.display(IPython.display.Video(video_url, width=600))
    return True


def print_multimodal_prompt(contents: list[str | Image | Part]):
    """
    Given contents that would be sent to Gemini,
    output the full multimodal prompt for ease of readability.
    """
    for content in contents:
        if display_content_as_image(content):
            continue
        if display_content_as_video(content):
            continue
        print(content)

In [17]:
# Task 4.2 Generate a video description
# In this cell, update the prompt to ask Gemini to describe the video URL referenced.
# You can use the documentation at the following link to assist.
# https://cloud.google.com/vertex-ai/docs/generative-ai/multimodal/sdk-for-gemini/gemini-sdk-overview-reference#generate-content-from-video
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/inference#sample-requests-text-stream-response
# Video URI: gs://github-repo/img/gemini/multimodality_usecases_overview/mediterraneansea.mp4

prompt = """
What is shown in this video?
Where should I go to see it?
What are the top 5 places in the world that look like this?
"""
video = Part.from_uri(
    file_uri="gs://github-repo/img/gemini/multimodality_usecases_overview/mediterraneansea.mp4",
    mime_type="video/mp4",
)
contents = [prompt, video]

responses = client.models.generate_content_stream(
    model=multimodal_model,
    contents=contents
)

print("-------Prompt--------")
print_multimodal_prompt(contents)

print("\n-------Response--------")
for response in responses:
    print(response.text, end="")

-------Prompt--------

What is shown in this video?
Where should I go to see it?
What are the top 5 places in the world that look like this?




-------Response--------
The video shows **Kaleiçi Marina**, the historic old harbor of **Antalya, Turkey**. 

### Where is it?
This specific location is the **Kaleiçi Old Town Harbor (Kaleiçi Yat Limanı)** in Antalya. It features a historic breakwater, a lighthouse, and is nestled at the base of dramatic limestone cliffs topped with modern and historic white buildings overlooking the turquoise waters of the Mediterranean Sea.

---

### Top 5 Places in the World That Look Like This
If you love the combination of dramatic coastal cliffs, historic harbor towns, and turquoise waters, here are the top five destinations in the world with a very similar aesthetic:

#### 1. **Amalfi Coast, Italy** (Specifically Positano or Amalfi)
* **Why it looks like this:** Famous for its sheer, dramatic cliffs dropping straight into the Mediterranean Sea. Colorful and white buildings are stacked precariously on the cliffside, overlooking bustling harbors filled with boats and yachts. 

#### 2. **Dubrovnik